# BimodalMath 75M on Colab G4 (Google AI Pro, 100 CU / month)

One notebook drives every session. Set `STAGE`, then **Runtime > Run all**. Each session trains up to
`MAX_WALL_MINUTES`, syncs `recovery.pt` / `best.pt` / `metrics.jsonl` / `status.json` every 20 minutes and on exit,
then disconnects the runtime so no idle compute units are burned.

## 100% Hugging Face setup (default `STORAGE_MODE = "hf"`)

No Google Drive at all. Code (`math-recogniser.zip`), `manifest.json`, `tokenizer.json` and the shards are pulled from
the private dataset repo `HF_SHARDS_REPO = "Wosca/math-recogniser-shards"`; checkpoints / resume state are written to
the private dataset repo `HF_REPO_ID = "Wosca/math-recogniser-runs"`.

1. Add your write-scoped Hugging Face token as **`HF_TOKEN`** in Colab's left sidebar **🔑 Secrets** (toggle
   **Notebook access** ON).
2. Check `HF_REPO_ID` / `HF_SHARDS_REPO` in Cell 1 (the runs repo is created on first run). Stage 2 shards are packed
   and uploaded by `colab/pack_stage2_colab_cpu.ipynb` on a Colab CPU runtime.
3. Select **Runtime → Change runtime type → G4 (RTX PRO 6000)** (or A100) and click **Runtime → Run all**.

On a 40 GB GPU (A100) Stage 2 needs a smaller micro-batch: `EXTRA_ARGS = ["--batch-size", "48", "--grad-accum", "2"]`.

### Fallbacks

`STORAGE_MODE = "drive_public_hf"` reads code and shards from the publicly shared Drive folder by file ID
(`https://drive.usercontent.google.com/download?id=...`, never `drive.mount()`) and writes runs to `HF_REPO_ID`. Drive
file IDs are embedded in Cell 1 (`EMBEDDED_DRIVE_IDS`) and refreshed from `drive_file_ids.json` when
`DRIVE_IDS_FILE_ID` is set. After adding files to Drive (e.g. `stage3_hard/`), re-run locally:

```
python tools/export_drive_ids.py --embed-notebook colab/train_bimodal_g4.ipynb
```

`STORAGE_MODE = "drive_mount"` keeps the original mounted-Drive workflow for accounts without Family Link.

| Stage | What | G4 time | Compute units |
|---|---|---|---|
| 1 | Vision + decoder pretraining (`stage1_train`) | ~3.2 h, 1 session | ~22 CU |
| 2 | End-to-end bimodal (`stage2_train`), clean-human cooldown | ~6 h, 2 x 3 h sessions | ~42 CU |
| 3 | Hard-example structure-reward GRPO (`stage3_hard`) | ~2.7 h, 1 session | ~18 CU |
| - | Reserve (restarts, re-validation, export) | | ~18 CU |

Session plan: Stage 1, then Stage 2 twice (the second run resumes automatically from Hugging Face / Drive), then mine
hard examples **locally** (0 CU) and run Stage 3:

```
python train_bimodal.py --mine-hard --shards <local shards> --init-from <stage2 best.pt>
```

Upload the resulting `stage3_hard/` folder and `hard_manifest.json` into `shards/` of `HF_SHARDS_REPO` (or
`math-recogniser-colab/shards/` on Drive) before Stage 3.

Shard layout (HF repo or Drive): `shards/` (manifest.json, tokenizer.json, stage1_train/, stage2_train/, val_dev/,
stage3_hard/) and `math-recogniser.zip`. Run outputs: `runs/<RUN_NAME>/stage<N>/` in `HF_REPO_ID` (or under
`DRIVE_ROOT` when mounted).

In [ ]:
# 1. Configuration, secrets and hardware check
STORAGE_MODE = "hf"                # "hf" (code + shards + runs on Hugging Face), "drive_public_hf" or "drive_mount"
STAGE = 1                          # 1, 2 or 3
MAX_WALL_MINUTES = 180             # session budget; training pauses cleanly and resumes next session
AUTO_DISCONNECT = True             # release the runtime after training (zero idle CUs)
RUN_NAME = "bimodal_75m_v1"
COMPILE = True
STAGE3_COPY_STAGE2 = False         # anchors already live inside stage3_hard/; enable only to stage stage2_train too
EXTRA_ARGS = []                    # e.g. ["--batch-size", "48", "--grad-accum", "2"] on a 40 GB A100
HF_REPO_ID = "Wosca/math-recogniser-runs"        # private dataset repo for checkpoints / resume state
HF_SHARDS_REPO = "Wosca/math-recogniser-shards"  # private dataset repo with math-recogniser.zip + shards/ ("hf")
HF_SQUASH_HISTORY = True           # drop superseded checkpoint versions at session end to save HF storage
DRIVE_IDS_FILE_ID = ""             # cloud ID of drive_file_ids.json; written by tools/export_drive_ids.py
DRIVE_ROOT = "/content/drive/MyDrive/math-recogniser-colab"   # drive_mount only
REPO_SOURCE = "zip"                # "zip" (math-recogniser.zip from HF / Drive), "drive_folder" (drive_mount) or git URL
REPO_BRANCH = "main"

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/math-recogniser")
ML_DIR = REPO_DIR / "ml"
DRIVE_SHARDS = Path(DRIVE_ROOT) / "shards"
LOCAL_SHARDS = Path("/content/shards")
DRIVE_RUNS = Path(DRIVE_ROOT) / "runs" / RUN_NAME
DRIVE_RUN_DIR = DRIVE_RUNS / f"stage{STAGE}"
LOCAL_RUN_DIR = Path("/content/runs") / RUN_NAME / f"stage{STAGE}"
HF_PREFIX = f"runs/{RUN_NAME}/stage{STAGE}"
assert STAGE in (1, 2, 3), "STAGE must be 1, 2 or 3"
assert STORAGE_MODE in ("hf", "drive_public_hf", "drive_mount"), \
    "STORAGE_MODE must be hf, drive_public_hf or drive_mount"

query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                       capture_output=True, text=True)
if query.returncode != 0:
    raise SystemExit("No GPU attached: Runtime > Change runtime type > G4")
print("nvidia-smi:", query.stdout.strip())

import torch

properties = torch.cuda.get_device_properties(0)
VRAM_GIB = properties.total_memory / 2**30
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}) | {properties.name} | "
      f"{VRAM_GIB:.1f} GiB | bf16 supported: {torch.cuda.is_bf16_supported()}")
if STAGE == 2 and VRAM_GIB < 50 and "--batch-size" not in [str(value) for value in EXTRA_ARGS]:
    print(f"HINT: {VRAM_GIB:.0f} GiB VRAM is tight for the Stage 2 default batch (128). Consider "
          'EXTRA_ARGS = ["--batch-size", "48", "--grad-accum", "2"] to avoid CUDA out-of-memory.')

if STORAGE_MODE in ("hf", "drive_public_hf"):
    HF_TOKEN = ""
    try:
        from google.colab import userdata

        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception as error:
        print(f"Colab Secrets: {type(error).__name__}: {error}")
    HF_TOKEN = (HF_TOKEN or os.environ.get("HF_TOKEN", "")).strip()
    for name, value in (("HF_REPO_ID", HF_REPO_ID), ("HF_SHARDS_REPO", HF_SHARDS_REPO)):
        if (name == "HF_REPO_ID" or STORAGE_MODE == "hf") and (
                value.count("/") != 1 or "<" in value or not all(value.split("/"))):
            raise SystemExit(f'Set {name} = "<your-hf-username>/<repo>" at the top of this cell.')
    if not HF_TOKEN.startswith("hf_"):
        raise SystemExit("HF_TOKEN missing: add a write-scoped Hugging Face token named HF_TOKEN in the left sidebar "
                         "Secrets (key icon) and switch Notebook access ON, then rerun.")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # explicit token for the private repos only, never for timm
    if STORAGE_MODE == "hf":
        print(f"Code + shards <- https://huggingface.co/datasets/{HF_SHARDS_REPO}")
    print(f"Run artifacts -> https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/{HF_PREFIX}")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive run directory:", DRIVE_RUN_DIR)

# BEGIN EMBEDDED_DRIVE_IDS (tools/export_drive_ids.py --embed-notebook)
EMBEDDED_DRIVE_IDS = {
    "math-recogniser.zip": {"id": "1Q9SgTEm90OF-CumFHWUoUj_kZ0wGoQPk", "size": 709333},
    "shards/audit_report.json": {"id": "1b2nb-ucEOuGaU7OJqVVXaNB43Q3LQfMj", "size": 989152},
    "shards/manifest.json": {"id": "1ZRSICWchGgvqCFyN6r-oNaDRd4Jy_T3z", "size": 112257},
    "shards/stage1_train/shard-00000.tar.gz": {"id": "1cxkvYRl12wckU0mgok7GVat-Eo56vnvB", "size": 38450701},
    "shards/stage1_train/shard-00001.tar.gz": {"id": "1ywk-iwmhfkD_q_3L4yk6rPYX_Viz2WXH", "size": 38576969},
    "shards/stage1_train/shard-00002.tar.gz": {"id": "16J_FMJQYNocMFaoQugPLvunkiGcFwRyP", "size": 38464474},
    "shards/stage1_train/shard-00003.tar.gz": {"id": "1zmVJCMmeVOEp9Twy4IH-YNooyXZlkFvp", "size": 38593313},
    "shards/stage1_train/shard-00004.tar.gz": {"id": "1v3GnM_Voyl1ilx1qPJ-7auMPF1UJqX4o", "size": 38422691},
    "shards/stage1_train/shard-00005.tar.gz": {"id": "1pzUsFSlqB-7nBWifWvcpaPdtLekLsLAK", "size": 38452344},
    "shards/stage1_train/shard-00006.tar.gz": {"id": "16qWKgqtgs7Ae0bm9dvu-26QH1g13_7wz", "size": 38515774},
    "shards/stage1_train/shard-00007.tar.gz": {"id": "1yjZ4k5HH_0F4iWePgOZYpJqzSkLbg8tV", "size": 38637040},
    "shards/stage1_train/shard-00008.tar.gz": {"id": "1gx0wk2a4GPaXtEDscgpV2htd_qDNw7oV", "size": 38814596},
    "shards/stage1_train/shard-00009.tar.gz": {"id": "1jUvTwhHzXuvpbE0nXkZ-ui77O7XsZdO_", "size": 38944464},
    "shards/stage1_train/shard-00010.tar.gz": {"id": "1gAGmJvkZ7qvhBhkyuDduASyxS1evcmCy", "size": 38725149},
    "shards/stage1_train/shard-00011.tar.gz": {"id": "1QLznVoIIkLvddAVZ1CxXFRXxb6JvhkXu", "size": 38804303},
    "shards/stage1_train/shard-00012.tar.gz": {"id": "1vjY_vsax-tNQiEmt2aP8V-0NcjoH4Fqy", "size": 38749970},
    "shards/stage1_train/shard-00013.tar.gz": {"id": "1NwmoPNe9RrI9TqNf50rLkK-vla43pkxF", "size": 39376578},
    "shards/stage1_train/shard-00014.tar.gz": {"id": "17PRxZYxQWBrTR-lgiXr0nXH_6caC-9qQ", "size": 39164214},
    "shards/stage1_train/shard-00015.tar.gz": {"id": "1kxXQXogRNcMrngJHbhgQbfN5sbOTrJEX", "size": 38464270},
    "shards/stage1_train/shard-00016.tar.gz": {"id": "1WQORXYoNXoUJFG4DMYKVJz1qur9-Hay-", "size": 38861050},
    "shards/stage1_train/shard-00017.tar.gz": {"id": "1en4I9d3lpBTW2ii9h841rQpf6ZW2lGNC", "size": 39199130},
    "shards/stage1_train/shard-00018.tar.gz": {"id": "1Z6IaWa31WWzZb7WNj2FUReRAeI4FmwEi", "size": 38402735},
    "shards/stage1_train/shard-00019.tar.gz": {"id": "17Jn6bnplJw-MXMR3HIGV1qCr7MqTnNH9", "size": 38687930},
    "shards/stage1_train/shard-00020.tar.gz": {"id": "1yR3B0nqaL8BCPnFoLWx-rcEOa03CM5tT", "size": 38650242},
    "shards/stage1_train/shard-00021.tar.gz": {"id": "1xH3lTjgQbBSQZ4H-ZrtxDWEVM1NhW5w8", "size": 38211025},
    "shards/stage1_train/shard-00022.tar.gz": {"id": "1O3TxqXhb1t89iISPnsacWtpizg2S5uV8", "size": 39076373},
    "shards/stage1_train/shard-00023.tar.gz": {"id": "1nQYbDEPvm9W9AoKik3iV0asTUTcIgPnW", "size": 38239240},
    "shards/stage1_train/shard-00024.tar.gz": {"id": "1afIjJOiGab8lkszzH0z9ujCFuBzdqwz1", "size": 39147205},
    "shards/stage1_train/shard-00025.tar.gz": {"id": "1UJAkrMq-a6kpcDHMFF680G9lWYYkpvra", "size": 38687730},
    "shards/stage1_train/shard-00026.tar.gz": {"id": "1Hs8pSBrSBdi4bEOazYM-WoVbDDCyp1KM", "size": 38659597},
    "shards/stage1_train/shard-00027.tar.gz": {"id": "1crtGKR6wLO0zgEfs5FMPUi70IjsjK3YL", "size": 38854471},
    "shards/stage1_train/shard-00028.tar.gz": {"id": "1M1oVY9E30R0wYbvtfuhLkIAkB9Q6oI5f", "size": 38641665},
    "shards/stage1_train/shard-00029.tar.gz": {"id": "1h4ljiNzvw7ImR_ojNV_5ZARP041Noi9o", "size": 39180688},
    "shards/stage1_train/shard-00030.tar.gz": {"id": "1R6hrdqo_3lxzBVzYCPLlh1iF_mE4EHrl", "size": 38994190},
    "shards/stage1_train/shard-00031.tar.gz": {"id": "16JfF08qNI0nVEdJM_lJmvN8px7pp3BpM", "size": 37921726},
    "shards/stage1_train/shard-00032.tar.gz": {"id": "1SiZn98-JJKRrCkm4XCmWW4ywu1FPaFIg", "size": 39037122},
    "shards/stage1_train/shard-00033.tar.gz": {"id": "1tIqylM5MQSiG1fFvkXbMlrlg_M_gZvDS", "size": 38356135},
    "shards/stage1_train/shard-00034.tar.gz": {"id": "1EgeSFWFrdG0pYel4PIBbyj-192iWoSrs", "size": 38497381},
    "shards/stage1_train/shard-00035.tar.gz": {"id": "1cwzACWHqD8wgfIc5W7OM1wloxwbxwEQf", "size": 38900333},
    "shards/stage1_train/shard-00036.tar.gz": {"id": "1ACVN5dvZtSA70vYbIIBuyQUcFfffAc72", "size": 38943424},
    "shards/stage1_train/shard-00037.tar.gz": {"id": "1Z3vJ4m2-DL0PjcSpTf9G8C1xtwMgkrFY", "size": 39213295},
    "shards/stage1_train/shard-00038.tar.gz": {"id": "1JpCKGZfPcMy8NdoULTaKhUGlbIRZqdMt", "size": 38799689},
    "shards/stage1_train/shard-00039.tar.gz": {"id": "19ZWd-YSkFTTxOx-_WjiwyiciD5MHvmhv", "size": 38325070},
    "shards/stage1_train/shard-00040.tar.gz": {"id": "193D6p3SUgFBFjzBIbK0n3H17xZoetR0t", "size": 38950006},
    "shards/stage1_train/shard-00041.tar.gz": {"id": "19i91bKOcWN8AZXoqeUju-C_MFP580qSU", "size": 38167092},
    "shards/stage1_train/shard-00042.tar.gz": {"id": "1TkU_klsNazYP5ULDdVtBZc_PaAOoIHcH", "size": 38896863},
    "shards/stage1_train/shard-00043.tar.gz": {"id": "1Q1t2jMuRa4lkV3B8RH7uj74kPP1OgxqE", "size": 38921139},
    "shards/stage1_train/shard-00044.tar.gz": {"id": "1KTPK97UtDH3jON2YS_xIGoZJZu4aGYAO", "size": 38369588},
    "shards/stage1_train/shard-00045.tar.gz": {"id": "1SFg9VDrjroTVBcSE6DJl9QzVE65J0hXw", "size": 38804070},
    "shards/stage1_train/shard-00046.tar.gz": {"id": "15J8-QGC2gj0q1u1HIJ5v8uDpvAfi2vR4", "size": 38567152},
    "shards/stage1_train/shard-00047.tar.gz": {"id": "14b2Kqo1jcg7bMU-JLdfrMZTILNurKha6", "size": 38864280},
    "shards/stage1_train/shard-00048.tar.gz": {"id": "11joBFow3koQXBOngC_o_YSxx1ingEHhk", "size": 38687281},
    "shards/stage1_train/shard-00049.tar.gz": {"id": "1f3d4xensehKPK2IYVGf_N0wJu5ZjMWyS", "size": 38832578},
    "shards/stage1_train/shard-00050.tar.gz": {"id": "1bX60wL6lo87ioS5ptEN1XDBHlOHXKqom", "size": 39025336},
    "shards/stage1_train/shard-00051.tar.gz": {"id": "1xnFrGXel7SeGhIQpRI0zz7aN2d9e_xA9", "size": 38319500},
    "shards/stage1_train/shard-00052.tar.gz": {"id": "1MZb49iXuS1o4Xsnmmi8RmTgOUHDPsb4B", "size": 38765913},
    "shards/stage1_train/shard-00053.tar.gz": {"id": "1_KMyRsDU7Q7iAlnHYibXEExw86yC-9G_", "size": 38875482},
    "shards/stage1_train/shard-00054.tar.gz": {"id": "1CTjkXYzBS8GoXBhjuL2ohqvAMYY0-CN8", "size": 38470491},
    "shards/stage1_train/shard-00055.tar.gz": {"id": "1T47rOYkcCcpWIwOz03jNlqXEaCJjkRkH", "size": 38631496},
    "shards/stage1_train/shard-00056.tar.gz": {"id": "1NKIJEqpPiuIFHIoEBa0soLu0pvMCRYHY", "size": 38679895},
    "shards/stage1_train/shard-00057.tar.gz": {"id": "1cCUz-dG5Sl8X3Cw9QBIab76H7GpO3qgI", "size": 38424897},
    "shards/stage1_train/shard-00058.tar.gz": {"id": "1n6sSrMWMv-UyjiCA_5WIEuzB9PiPRuYB", "size": 38284778},
    "shards/stage1_train/shard-00059.tar.gz": {"id": "1nclIp5Fg5eWgvBt6C3MQYw0KTpbScEh1", "size": 38906316},
    "shards/stage1_train/shard-00060.tar.gz": {"id": "1Omo9g3fCIs34_fZYwFyNItoQN9gJiewM", "size": 38233045},
    "shards/stage1_train/shard-00061.tar.gz": {"id": "1xi-XCsUpw4Jzycme-hC0WfcUiqyk8V58", "size": 38398928},
    "shards/stage1_train/shard-00062.tar.gz": {"id": "1yNEQcdKBZ6526NElvVSn4v1gG2XeAkZo", "size": 38745430},
    "shards/stage1_train/shard-00063.tar.gz": {"id": "1dhGkkiY9NY5oI2Bvkpm6DNimNoZoVXcS", "size": 38674839},
    "shards/stage1_train/shard-00064.tar.gz": {"id": "1HdxshFCAcdYeWxSVxGlt-_fslWkYCVT2", "size": 38750610},
    "shards/stage1_train/shard-00065.tar.gz": {"id": "1y9n85U1ie4tTcEhQ0UvnR3NwAgTW868_", "size": 38493342},
    "shards/stage1_train/shard-00066.tar.gz": {"id": "1XDJEilpPILs-_S5wlwGjsfa0knrx3ing", "size": 38617374},
    "shards/stage1_train/shard-00067.tar.gz": {"id": "1e9ZNh8-Er8Qc1WaLtngZNtuOQFKvOdcD", "size": 38365777},
    "shards/stage1_train/shard-00068.tar.gz": {"id": "1qdR261GNTRp-RLqUqerWfPMF4Wgy3UhS", "size": 38610615},
    "shards/stage1_train/shard-00069.tar.gz": {"id": "1LC2HuLY21voFRaFLIN5WyzKJTRV4nGq3", "size": 38429322},
    "shards/stage1_train/shard-00070.tar.gz": {"id": "1WHnqEeP8VgHGuEwpl9_eNMNWhfcgyzpu", "size": 38594829},
    "shards/stage1_train/shard-00071.tar.gz": {"id": "1B4i-lHtPDU6T8ZsEJrh0Db17wiZSGGUa", "size": 38981696},
    "shards/stage1_train/shard-00072.tar.gz": {"id": "1JrNv3-0q_iyh9cw2DWlGOZLAC4JCD_yB", "size": 38580096},
    "shards/stage1_train/shard-00073.tar.gz": {"id": "1uM9kw7kxJRdg_nZ7Xv-JIJbzQTv0FcMN", "size": 38727012},
    "shards/stage1_train/shard-00074.tar.gz": {"id": "1NVFA1PK3K_oGZf0koYLGZSpVLum_bodE", "size": 38004375},
    "shards/stage1_train/shard-00075.tar.gz": {"id": "1-GLQ7icWgoi3-rGOVHzWNsZ-ztVlkZ5u", "size": 39463421},
    "shards/stage1_train/shard-00076.tar.gz": {"id": "1qjLVuGvfWi2TjqrDtsj_pem74Fcv_hVN", "size": 38533167},
    "shards/stage1_train/shard-00077.tar.gz": {"id": "1SXdGlROhmzlmGEWaTBoPU57ESEN9w4l_", "size": 38551966},
    "shards/stage1_train/shard-00078.tar.gz": {"id": "1ojekEWBOMRkythgDw829fgPb0FrW6JF6", "size": 38675916},
    "shards/stage1_train/shard-00079.tar.gz": {"id": "1A2O7Jcle8sYtHAtu1Zq70wi2qzwjuLFR", "size": 39253653},
    "shards/stage1_train/shard-00080.tar.gz": {"id": "12wBpdEusDT0KuVfDNixunCZ8yfTCm2dC", "size": 38140096},
    "shards/stage1_train/shard-00081.tar.gz": {"id": "16PW0JiUH5FrzziFrr6lnAv27mvRVRGfN", "size": 38469034},
    "shards/stage1_train/shard-00082.tar.gz": {"id": "1Sfsx1UwjN5Rwv9w8uo0dfyvfHLhR5Hqw", "size": 38682988},
    "shards/stage1_train/shard-00083.tar.gz": {"id": "1iCPWVgpSZxeFxGjnyOyYhV0tpV6IJahO", "size": 39294416},
    "shards/stage1_train/shard-00084.tar.gz": {"id": "1PI5w3MTAIUfaUuoFjInrLOyPXRxSfBOg", "size": 38774079},
    "shards/stage1_train/shard-00085.tar.gz": {"id": "1Gtyo976trWp2UEws-e2ZzLe8nKsODnwb", "size": 38671226},
    "shards/stage1_train/shard-00086.tar.gz": {"id": "1MTg6h_cgYwRvsiIlQDKvz05TEJ6sF_Ze", "size": 38345458},
    "shards/stage1_train/shard-00087.tar.gz": {"id": "1N8NRjcSJSnbFzy5GeEeNMxCEvsfylyyT", "size": 39187588},
    "shards/stage1_train/shard-00088.tar.gz": {"id": "1xTyjZEcVfVNXRJdmihvmQt4FrardUC8G", "size": 38852936},
    "shards/stage1_train/shard-00089.tar.gz": {"id": "1-hKra_luxo6oU9Vm53MYzg0MK0J_cUsU", "size": 38641925},
    "shards/stage1_train/shard-00090.tar.gz": {"id": "1JRZSISEZVMClNKju2dr0wPF1D6DWhr0B", "size": 38360427},
    "shards/stage1_train/shard-00091.tar.gz": {"id": "11VX4_0gQ0A1MV9phs8EoKza0UeMePB_P", "size": 38487944},
    "shards/stage1_train/shard-00092.tar.gz": {"id": "15AIMwNuGA7gegFDxbM7LtC-Ci9D5DRyO", "size": 38643860},
    "shards/stage1_train/shard-00093.tar.gz": {"id": "19PdjbemL0zRCGIPNek-3521LGhfZmGGT", "size": 38249265},
    "shards/stage1_train/shard-00094.tar.gz": {"id": "1sIeGGu2_-P-BFrX7RMD0Ovl_5yoR99h-", "size": 38587571},
    "shards/stage1_train/shard-00095.tar.gz": {"id": "1GtI3nMU6xAMp21QYDSUj8wxV0fKyxKJz", "size": 39303856},
    "shards/stage1_train/shard-00096.tar.gz": {"id": "1ML2NN3KIVGDhFPGQ6rBL7gnmzo0NiHfo", "size": 39074027},
    "shards/stage1_train/shard-00097.tar.gz": {"id": "1MNUDXDz2nzaBm6HKmQqmqCAS8yjnUIDV", "size": 39043911},
    "shards/stage1_train/shard-00098.tar.gz": {"id": "1GckT5euEsq158VnzvPjKukRCq22Ph8kY", "size": 39850981},
    "shards/stage1_train/shard-00099.tar.gz": {"id": "11z2IPiBLU6mryhLYZkaCEPsUECojb40j", "size": 38414267},
    "shards/stage1_train/shard-00100.tar.gz": {"id": "1341ouT3iyTQmS4NI3l64_JEEvj9hTfc-", "size": 38557434},
    "shards/stage1_train/shard-00101.tar.gz": {"id": "1vdrBqMnNy3gAfLWx1O7WMwvEjL8rN1Le", "size": 38674409},
    "shards/stage1_train/shard-00102.tar.gz": {"id": "1UvGrhEt-M3eI-uR1WLZYAlI8RDus5nu6", "size": 38939005},
    "shards/stage1_train/shard-00103.tar.gz": {"id": "17v8EYyB5VOXz35K6vhdMSyZ5qLj-aruW", "size": 38281422},
    "shards/stage1_train/shard-00104.tar.gz": {"id": "1QIaZGP9Cq828GbTJh-wF83moZFn3Bz1G", "size": 38845834},
    "shards/stage1_train/shard-00105.tar.gz": {"id": "1dPl8jOiPqCRyyqRFVkJ1dtoTcSxf1i1T", "size": 38813639},
    "shards/stage1_train/shard-00106.tar.gz": {"id": "1Pzdq2M1jRWs6uKRXBJ2N67xHamzqAC2F", "size": 38783828},
    "shards/stage1_train/shard-00107.tar.gz": {"id": "1YAl1bSOli4bFdbaLfqoFU6Ob9v_4F7an", "size": 38346228},
    "shards/stage1_train/shard-00108.tar.gz": {"id": "1SbQOgrsDjuricF_Smji-mr7t5IMklcXr", "size": 38452595},
    "shards/stage1_train/shard-00109.tar.gz": {"id": "1OBkiQGGgQKkHtvARytCrAh69FZccLMHc", "size": 38684952},
    "shards/stage1_train/shard-00110.tar.gz": {"id": "1jfq-g_3amtooAfsutNZTf5S_-s8BK69p", "size": 38797415},
    "shards/stage1_train/shard-00111.tar.gz": {"id": "16Hz822UOY0vxjqTu6zQDAkHUBcNErdVN", "size": 38521723},
    "shards/stage1_train/shard-00112.tar.gz": {"id": "19PEalVVlxxd1In_M3nRxnOG03gX7BE6p", "size": 38938543},
    "shards/stage1_train/shard-00113.tar.gz": {"id": "17yhI_KXQD3f-BaC4x0GNdqaMmzolLqHa", "size": 38971727},
    "shards/stage1_train/shard-00114.tar.gz": {"id": "16CwwpsppXNcQ3r64iVJTf_pnkyd59zpq", "size": 38344504},
    "shards/stage1_train/shard-00115.tar.gz": {"id": "1ZZTg71aZxV9uut_vLWvMF5oL6a8vE6zc", "size": 38897780},
    "shards/stage1_train/shard-00116.tar.gz": {"id": "1ax-4BB1WzuWPiRoPUHd53W4r4adk6j9T", "size": 38965450},
    "shards/stage1_train/shard-00117.tar.gz": {"id": "1939lRNiaUHRMqjjKlHMCVn45dtofNGr_", "size": 38478516},
    "shards/stage1_train/shard-00118.tar.gz": {"id": "1_RVoL6-Zd1yp9dpzs0lNdpEeXQeCNSE2", "size": 38341378},
    "shards/stage1_train/shard-00119.tar.gz": {"id": "1e5xr25ZWuFXkaWKaRUVIuK-I54JX9Ruw", "size": 38362214},
    "shards/stage1_train/shard-00120.tar.gz": {"id": "1-1vFPE8UVfnLkIuWt5PwsPOPUiODkjPE", "size": 38563947},
    "shards/stage1_train/shard-00121.tar.gz": {"id": "15wBpfrHkFYirto8BvxYMlptvIGC5lWUH", "size": 31201070},
    "shards/tokenizer.json": {"id": "13J49Gmc3KrQQKwIyUjkGXkRvlt1P6HPQ", "size": 14959},
    "shards/val_dev/shard-00000.tar.gz": {"id": "1sJfcahsElAwJ2JnJu0CFfjVTXlSiUDLo", "size": 5950200},
}
# END EMBEDDED_DRIVE_IDS

In [ ]:
# 2. Repository and dependencies (Colab's preinstalled CUDA PyTorch is never touched)
import random
import shutil
import time
import urllib.error
import urllib.request
import zipfile

DRIVE_DOWNLOAD_URL = "https://drive.usercontent.google.com/download?id={}&export=download&confirm=t"


def sh(*command, cwd=None):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], check=True, cwd=cwd)


sh(sys.executable, "-m", "pip", "install", "-q", "timm>=1.0", "pyyaml", "antlr4-python3-runtime>=4.11,<4.12",
   "huggingface_hub>=0.24")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # public timm weights never need (or break on) a token
if STORAGE_MODE == "drive_mount":
    os.environ.pop("HF_TOKEN", None)


def download_drive_file(file_id, destination, expected_magic=None, expected_size=None, retries=5):
    """Public Drive download with atomic .tmp -> os.replace; HTML interstitials fail the magic check and retry."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    request = urllib.request.Request(DRIVE_DOWNLOAD_URL.format(file_id), headers={"User-Agent": "Mozilla/5.0"})
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(request, timeout=120) as response, open(temporary, "wb") as handle:
                shutil.copyfileobj(response, handle, length=1 << 22)
            with open(temporary, "rb") as handle:
                head = handle.read(64)
            if expected_magic and not head.startswith(expected_magic):
                raise ValueError(f"unexpected content {head[:40]!r} (Drive interstitial or quota page?)")
            size = temporary.stat().st_size
            if expected_size is not None and size != int(expected_size):
                raise ValueError(f"got {size} bytes, expected {expected_size}")
            os.replace(temporary, destination)
            return destination
        except (OSError, ValueError) as error:
            temporary.unlink(missing_ok=True)
            if attempt == retries:
                raise RuntimeError(f"Drive download {file_id} -> {destination.name} failed: {error}") from error
            delay = min(60.0, 2.0 ** attempt + random.random())
            print(f"  retry {attempt}/{retries - 1} for {destination.name} in {delay:.0f}s: {error}", flush=True)
            time.sleep(delay)


DRIVE_IDS = dict(EMBEDDED_DRIVE_IDS)
if STORAGE_MODE == "drive_public_hf":
    if DRIVE_IDS_FILE_ID:
        try:
            fresh = download_drive_file(DRIVE_IDS_FILE_ID, Path("/content/drive_file_ids.json"), b"{", retries=3)
            DRIVE_IDS.update(json.loads(fresh.read_text(encoding="utf-8"))["files"])
            print(f"drive_file_ids.json: {len(DRIVE_IDS)} Drive IDs")
        except Exception as error:
            print(f"drive_file_ids.json unavailable ({error}); using {len(DRIVE_IDS)} embedded IDs")
    else:
        print(f"using {len(DRIVE_IDS)} embedded Drive IDs")


def drive_entry(path):
    entry = DRIVE_IDS.get(path)
    if not entry:
        raise SystemExit(f"No Drive ID for {path}: run tools/export_drive_ids.py --embed-notebook locally")
    return entry


IGNORE = shutil.ignore_patterns(".git", ".venv", "__pycache__", "node_modules", "checkpoints", ".smoke-checkpoints",
                                ".pytest_cache", "*.pt")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if REPO_SOURCE in ("zip", "drive_zip"):
    if STORAGE_MODE == "hf":
        from huggingface_hub import hf_hub_download

        archive_path = Path(hf_hub_download(repo_id=HF_SHARDS_REPO, filename="math-recogniser.zip",
                                            repo_type="dataset", token=HF_TOKEN, force_download=True))
        print(f"math-recogniser.zip <- hf://datasets/{HF_SHARDS_REPO} ({archive_path.stat().st_size:,} bytes)")
    elif STORAGE_MODE == "drive_public_hf":
        archive_path = download_drive_file(drive_entry("math-recogniser.zip")["id"],
                                           Path("/content/math-recogniser.zip"), b"PK\x03\x04")
    else:
        archive_path = Path(DRIVE_ROOT) / "math-recogniser.zip"
    unpacked = Path("/content/_repo_unpack")
    shutil.rmtree(unpacked, ignore_errors=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpacked)
    roots = [path for path in unpacked.iterdir() if path.is_dir() and (path / "ml").is_dir()]
    shutil.move(str(roots[0] if roots else unpacked), str(REPO_DIR))
    shutil.rmtree(unpacked, ignore_errors=True)
elif REPO_SOURCE == "drive_folder":
    if STORAGE_MODE != "drive_mount":
        raise SystemExit('REPO_SOURCE = "drive_folder" needs STORAGE_MODE = "drive_mount"')
    shutil.copytree(Path(DRIVE_ROOT) / "math-recogniser", REPO_DIR, ignore=IGNORE)
else:
    sh("git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_SOURCE, REPO_DIR)
assert (ML_DIR / "train_bimodal.py").is_file(), f"train_bimodal.py not found under {ML_DIR}"
TRAIN_SOURCE = (ML_DIR / "train_bimodal.py").read_text(encoding="utf-8")
if STORAGE_MODE != "drive_mount" and '"--hf-sync-repo"' not in TRAIN_SOURCE:
    raise SystemExit("math-recogniser.zip predates Hugging Face sync: re-upload it (Google Drive may still be "
                     "uploading the new version; see drive.google.com > Manage versions). Rerun once it is live.")
if "--grad-accum" in [str(value) for value in EXTRA_ARGS] and '"--grad-accum"' not in TRAIN_SOURCE:
    raise SystemExit("math-recogniser.zip predates --grad-accum: re-upload the zip, then rerun.")
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import timm

print("timm", timm.__version__, "| torch", torch.__version__, "| repo", REPO_DIR)

In [ ]:
# 3. Stage only the shards this stage needs onto local NVMe (parallel download / decompress), then verify SHA-256
import gzip
from concurrent.futures import ThreadPoolExecutor

from data.bimodal_shards import sha256_file, verify_manifest_checksums

NEEDED = {1: ["stage1_train"], 2: ["stage2_train"], 3: ["stage3_hard"]}[STAGE] + ["val_dev"]
if STAGE == 3 and STAGE3_COPY_STAGE2:
    NEEDED.append("stage2_train")
FILES = ["manifest.json", "tokenizer.json"] + (["hard_manifest.json"] if STAGE == 3 else [])


def stage_plain_file(source: Path, destination: Path) -> bool:
    if destination.is_file() and destination.stat().st_size == source.stat().st_size:
        return False
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    shutil.copyfile(source, temporary)
    os.replace(temporary, destination)
    return True


if STORAGE_MODE == "hf":
    from huggingface_hub import HfApi, hf_hub_download

    HF_SHARD_FILES = set(HfApi(token=HF_TOKEN).list_repo_files(HF_SHARDS_REPO, repo_type="dataset"))
    print(f"{HF_SHARDS_REPO}: {len(HF_SHARD_FILES)} files", flush=True)


def download_hf_file(path_in_repo: str, destination: Path, retries: int = 4) -> Path:
    """hf_hub_download into a private staging dir (no ~/.cache copy), then atomic os.replace."""
    if path_in_repo not in HF_SHARD_FILES:
        raise SystemExit(f"hf://datasets/{HF_SHARDS_REPO}/{path_in_repo} does not exist")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    staging = destination.parent / f".hf-download-{destination.name}"
    try:
        for attempt in range(1, retries + 1):
            try:
                downloaded = hf_hub_download(repo_id=HF_SHARDS_REPO, filename=path_in_repo, repo_type="dataset",
                                             token=HF_TOKEN, local_dir=staging)
                os.replace(downloaded, destination)
                return destination
            except Exception as error:  # noqa: BLE001
                if attempt == retries:
                    raise RuntimeError(f"HF download {path_in_repo} failed: {error}") from error
                print(f"  retry {attempt}/{retries - 1} for {path_in_repo}: {type(error).__name__}: {error}",
                      flush=True)
                time.sleep(min(30.0, 2.0 ** attempt))
    finally:
        shutil.rmtree(staging, ignore_errors=True)


started = time.time()
for name in FILES:
    if STORAGE_MODE == "hf":
        download_hf_file(f"shards/{name}", LOCAL_SHARDS / name)
    elif STORAGE_MODE == "drive_public_hf":
        download_drive_file(drive_entry(f"shards/{name}")["id"], LOCAL_SHARDS / name, b"{")
    else:
        source = DRIVE_SHARDS / name
        if not source.is_file():
            raise SystemExit(f"Missing {source}")
        stage_plain_file(source, LOCAL_SHARDS / name)

# Expected uncompressed .tar byte sizes from the manifests (skip already-staged shards on reruns)
expected_sizes: dict[str, int] = {}
manifest_data = json.loads((LOCAL_SHARDS / "manifest.json").read_text(encoding="utf-8"))
for info in manifest_data.get("stages", {}).values():
    for shard in info.get("shards", []):
        if "bytes" in shard:
            expected_sizes[f"{info['directory']}/{shard['file']}"] = int(shard["bytes"])
if STAGE == 3:
    hard_data = json.loads((LOCAL_SHARDS / "hard_manifest.json").read_text(encoding="utf-8"))
    for shard in hard_data.get("shards", []):
        if "bytes" in shard:
            expected_sizes[f"{hard_data['directory']}/{shard['file']}"] = int(shard["bytes"])


def already_staged(destination: Path, directory: str) -> bool:
    want = expected_sizes.get(f"{directory}/{destination.name}")
    return destination.is_file() and want is not None and destination.stat().st_size == want


def decompress(archive: Path, destination: Path) -> None:
    temporary = destination.with_name(destination.name + ".tmp")
    with gzip.open(archive, "rb") as source, open(temporary, "wb") as target:
        shutil.copyfileobj(source, target, length=1 << 22)
    os.replace(temporary, destination)


def stage_drive_shard(name: str, entry: dict, directory: str) -> tuple[bool, int]:
    is_gz = name.endswith(".tar.gz")
    destination = LOCAL_SHARDS / directory / (name[:-3] if is_gz else name)
    if already_staged(destination, directory):
        return False, destination.stat().st_size
    if not is_gz:
        download_drive_file(entry["id"], destination, expected_size=entry.get("size"))
        return True, destination.stat().st_size
    archive = LOCAL_SHARDS / directory / name
    try:
        download_drive_file(entry["id"], archive, b"\x1f\x8b", entry.get("size"))
        decompress(archive, destination)
    finally:
        archive.unlink(missing_ok=True)
    return True, destination.stat().st_size


def stage_hf_shard(name: str, directory: str) -> tuple[bool, int]:
    is_gz = name.endswith(".tar.gz")
    destination = LOCAL_SHARDS / directory / (name[:-3] if is_gz else name)
    if already_staged(destination, directory):
        return False, destination.stat().st_size
    if not is_gz:
        download_hf_file(f"shards/{directory}/{name}", destination)
        return True, destination.stat().st_size
    archive = LOCAL_SHARDS / directory / name
    try:
        download_hf_file(f"shards/{directory}/{name}", archive)
        decompress(archive, destination)
    finally:
        archive.unlink(missing_ok=True)
    return True, destination.stat().st_size


def stage_mounted_shard(source: Path, directory: str) -> tuple[bool, int]:
    is_gz = source.name.endswith(".tar.gz")
    destination = LOCAL_SHARDS / directory / (source.name[:-3] if is_gz else source.name)
    if already_staged(destination, directory) or (
            not is_gz and destination.is_file() and destination.stat().st_size == source.stat().st_size):
        return False, destination.stat().st_size
    destination.parent.mkdir(parents=True, exist_ok=True)
    if is_gz:
        decompress(source, destination)
    else:
        stage_plain_file(source, destination)
    return True, destination.stat().st_size


tasks = []
for directory in NEEDED:
    if STORAGE_MODE == "hf":
        prefix = f"shards/{directory}/"
        names = {path[len(prefix):] for path in HF_SHARD_FILES
                 if path.startswith(prefix) and path.endswith((".tar.gz", ".tar"))
                 and path[len(prefix):].startswith("shard-") and "/" not in path[len(prefix):]}
        listed = {key.split("/", 1)[1] for key in expected_sizes if key.startswith(f"{directory}/")}
        if listed:  # stale repo shards outside the manifest would otherwise be globbed into training
            names = {name for name in names if (name[:-3] if name.endswith(".gz") else name) in listed}
        chosen = sorted(name for name in names if name.endswith(".tar.gz")) or sorted(names)
        if not chosen:
            raise SystemExit(f"No {prefix}shard-*.tar.gz in hf://datasets/{HF_SHARDS_REPO}: upload them first")
        (LOCAL_SHARDS / directory).mkdir(parents=True, exist_ok=True)
        tasks.extend((stage_hf_shard, (name, directory)) for name in chosen)
    elif STORAGE_MODE == "drive_public_hf":
        prefix = f"shards/{directory}/"
        entries = {path[len(prefix):]: entry for path, entry in DRIVE_IDS.items()
                   if path.startswith(prefix) and path.endswith((".tar.gz", ".tar"))
                   and path[len(prefix):].startswith("shard-") and "/" not in path[len(prefix):]}
        gz = sorted(name for name in entries if name.endswith(".tar.gz"))
        chosen = gz or sorted(entries)
        if not chosen:
            raise SystemExit(f"No Drive IDs for {prefix}shard-*: upload it, then run tools/export_drive_ids.py")
        (LOCAL_SHARDS / directory).mkdir(parents=True, exist_ok=True)
        tasks.extend((stage_drive_shard, (name, entries[name], directory)) for name in chosen)
    else:
        source_dir = DRIVE_SHARDS / directory
        chosen = sorted(source_dir.glob("shard-*.tar.gz")) or sorted(source_dir.glob("shard-*.tar"))
        if not chosen:
            raise SystemExit(f"No shard-*.tar.gz or shard-*.tar files in {source_dir}")
        tasks.extend((stage_mounted_shard, (path, directory)) for path in chosen)
    print(f"  {directory}: {len(chosen)} shard(s) [{'.tar.gz' if chosen and str(chosen[0]).endswith('.gz') else '.tar'}]",
          flush=True)

copied = skipped = total_bytes = 0
with ThreadPoolExecutor(max_workers=8) as pool:
    for did_stage, nbytes in pool.map(lambda task: task[0](*task[1]), tasks):
        total_bytes += nbytes
        copied += did_stage
        skipped += not did_stage

print(f"staged {copied} shard(s), {skipped} already present, {total_bytes / 2**30:.2f} GiB uncompressed "
      f"in {time.time() - started:.0f}s -> {LOCAL_SHARDS}", flush=True)

results = {key: ok for key, ok in verify_manifest_checksums(LOCAL_SHARDS).items() if key.split("/")[0] in NEEDED}
if STAGE == 3:
    hard = json.loads((LOCAL_SHARDS / "hard_manifest.json").read_text(encoding="utf-8"))
    for shard in hard["shards"]:
        path = LOCAL_SHARDS / hard["directory"] / shard["file"]
        results[f"{hard['directory']}/{shard['file']}"] = path.is_file() and sha256_file(path) == shard["sha256"]
bad = [key for key, ok in results.items() if not ok]
if bad:
    raise SystemExit(f"Checksum mismatch: {bad[:10]}")
print(f"verified {len(results)} shard checksum(s) OK")

In [ ]:
# 4. Train with auto-resume (HF dataset repo or Drive) and 20-minute atomic sync
command = [sys.executable, "-u", "train_bimodal.py", "--stage", str(STAGE), "--shards", str(LOCAL_SHARDS),
           "--output", str(LOCAL_RUN_DIR), "--resume", "auto", "--max-wall-minutes", str(MAX_WALL_MINUTES),
           "--sync-every-minutes", "20"]
if STORAGE_MODE in ("hf", "drive_public_hf"):
    command += ["--hf-sync-repo", HF_REPO_ID, "--hf-sync-prefix", HF_PREFIX]
    if HF_SQUASH_HISTORY:
        command.append("--hf-squash-history")
    if STAGE > 1:
        command += ["--hf-init-path", f"runs/{RUN_NAME}/stage{STAGE - 1}/best.pt"]
else:
    recovery = DRIVE_RUN_DIR / "recovery.pt"
    command += ["--drive-sync-dir", str(DRIVE_RUN_DIR)]
    if STAGE > 1:
        previous = DRIVE_RUNS / f"stage{STAGE - 1}" / "best.pt"
        if previous.is_file():
            command += ["--init-from", str(previous)]
        elif not recovery.is_file():
            raise SystemExit(f"Stage {STAGE} starts from {previous}, which does not exist yet")
    print("resuming from Drive recovery.pt" if recovery.is_file() else "fresh start")
if COMPILE:
    command.append("--compile")
command += [str(value) for value in EXTRA_ARGS]
print("$", " ".join(command), flush=True)

process = subprocess.Popen(command, cwd=ML_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
for line in process.stdout:
    print(line, end="")
TRAIN_EXIT_CODE = process.wait()
print(f"train_bimodal.py exited with code {TRAIN_EXIT_CODE}")

In [ ]:
# 5. Status check, then zero-idle auto-disconnect
import time

HF_RUNS = STORAGE_MODE in ("hf", "drive_public_hf")
RUN_DIR = LOCAL_RUN_DIR if HF_RUNS else DRIVE_RUN_DIR
CHECKPOINTS = (f"https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/{HF_PREFIX}"
               if HF_RUNS else str(DRIVE_RUN_DIR))
keep_runtime = False
try:
    exit_code = globals().get("TRAIN_EXIT_CODE")
    if HF_RUNS and exit_code == 75:
        print("Final HF upload failed inside training; retrying once from the notebook...", flush=True)
        try:
            from train_bimodal import HFSync

            keep_runtime = not HFSync(LOCAL_RUN_DIR, HF_REPO_ID, HF_PREFIX, 0, HF_TOKEN).sync(force=True)
        except Exception as error:
            print(f"HF upload retry failed: {type(error).__name__}: {error}")
            keep_runtime = True
    status_path = RUN_DIR / "status.json"
    status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.is_file() else {}
    print(json.dumps(status, indent=2))
    metrics_path = RUN_DIR / "metrics.jsonl"
    if metrics_path.is_file():
        for line in metrics_path.read_text(encoding="utf-8").splitlines()[-5:]:
            record = json.loads(line)
            val = record.get("val") or {}
            print(f"epoch {record['epoch']} step {record['global_step']} loss {record['train'].get('loss', float('nan')):.4f} "
                  f"val normalized_exact {val.get('normalized_exact', float('nan')):.4f} "
                  f"token_cer {val.get('token_cer', float('nan')):.4f} cooldown={record.get('cooldown')}")
    state = status.get("state")
    print("Checkpoints:", CHECKPOINTS)
    if state == "paused_wall_limit":
        print("Paused at the wall limit: run this notebook again (same STAGE) to resume.")
    elif state == "completed":
        print(f"Stage {STAGE} completed. best.pt: {CHECKPOINTS}/best.pt")
    else:
        print(f"Training state {state!r} (exit code {exit_code}); inspect the log above.")
finally:
    if keep_runtime:
        print(f"NOT disconnecting: artifacts exist only in {LOCAL_RUN_DIR}. Rerun this cell or download them.")
    elif AUTO_DISCONNECT:
        print("Disconnecting the runtime in 15 seconds (set AUTO_DISCONNECT = False to keep it)...", flush=True)
        time.sleep(15)
        if STORAGE_MODE == "drive_mount":
            try:
                drive.flush_and_unmount()
            except Exception as error:
                print("drive flush failed:", error)
        from google.colab import runtime

        runtime.unassign()